In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonRaphson(
    model=model,
    lr=1,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.6036729216575623
epoch:  1, loss: 0.5147436857223511
epoch:  2, loss: 0.41513141989707947
epoch:  3, loss: 0.31870603561401367
epoch:  4, loss: 0.313363641500473
epoch:  5, loss: 0.265941321849823
epoch:  6, loss: 0.08134477585554123
epoch:  7, loss: 0.021595504134893417
epoch:  8, loss: 0.019892917945981026
epoch:  9, loss: 0.019783485680818558
epoch:  10, loss: 0.01123960129916668
epoch:  11, loss: 0.010769803076982498
epoch:  12, loss: 0.010338705964386463
epoch:  13, loss: 0.009552910923957825
epoch:  14, loss: 0.006800523959100246
epoch:  15, loss: 0.005822448059916496
epoch:  16, loss: 0.005167735740542412
epoch:  17, loss: 0.004706851206719875
epoch:  18, loss: 0.004341945983469486
epoch:  19, loss: 0.004077363293617964
epoch:  20, loss: 0.003889190498739481
epoch:  21, loss: 0.0037536537274718285
epoch:  22, loss: 0.0035881665535271168
epoch:  23, loss: 0.0034698941744863987
epoch:  24, loss: 0.003455115482211113
epoch:  25, loss: 0.0034430017694830894
epoch:

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.935377836227417
Test metrics:  R2 = 0.5503177642822266


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonRaphson(
    model=model,
    lr=1,
    c1=1e-4,
    c2=0.9,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="wolfe"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.1385185420513153
epoch:  1, loss: 0.1362878531217575
epoch:  2, loss: 0.13378416001796722
epoch:  3, loss: 0.10396553575992584
epoch:  4, loss: 0.10225006192922592
epoch:  5, loss: 0.10057953745126724
epoch:  6, loss: 0.0797758549451828
epoch:  7, loss: 0.06594186276197433
epoch:  8, loss: 0.05474646016955376
epoch:  9, loss: 0.0458562932908535
epoch:  10, loss: 0.03895990923047066
epoch:  11, loss: 0.033357638865709305
epoch:  12, loss: 0.02873053215444088
epoch:  13, loss: 0.02497720718383789
epoch:  14, loss: 0.02180471457540989
epoch:  15, loss: 0.01916043646633625
epoch:  16, loss: 0.016902999952435493
epoch:  17, loss: 0.015036729164421558
epoch:  18, loss: 0.013450877740979195
epoch:  19, loss: 0.013314666226506233
epoch:  20, loss: 0.012014675885438919
epoch:  21, loss: 0.010891263373196125
epoch:  22, loss: 0.009949584491550922
epoch:  23, loss: 0.009132172912359238
epoch:  24, loss: 0.009059018455445766
epoch:  25, loss: 0.008349846117198467
epoch:  26, los

In [8]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.919345498085022
Test metrics:  R2 = 0.8484064340591431


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonRaphson(
    model=model,
    lr=1,
    c1=1e-4,
    c2=0.9,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="strong-wolfe"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.6504327058792114
epoch:  1, loss: 0.591343343257904
epoch:  2, loss: 0.5445452928543091
epoch:  3, loss: 0.5445452928543091
epoch:  4, loss: 0.5445452928543091
epoch:  5, loss: 0.5445452928543091
epoch:  6, loss: 0.5445452928543091
epoch:  7, loss: 0.5445452928543091
epoch:  8, loss: 0.5445452928543091
epoch:  9, loss: 0.5445452928543091
epoch:  10, loss: 0.5445452928543091
epoch:  11, loss: 0.5445452928543091
epoch:  12, loss: 0.5445452928543091
epoch:  13, loss: 0.5445452928543091
epoch:  14, loss: 0.5445452928543091
epoch:  15, loss: 0.5445452928543091
epoch:  16, loss: 0.5445452928543091
epoch:  17, loss: 0.5445452928543091
epoch:  18, loss: 0.5445452928543091
epoch:  19, loss: 0.5445452928543091
epoch:  20, loss: 0.5445452928543091
epoch:  21, loss: 0.5445452928543091
epoch:  22, loss: 0.5445452928543091
epoch:  23, loss: 0.5445452928543091
epoch:  24, loss: 0.5445452928543091
epoch:  25, loss: 0.5445452928543091
epoch:  26, loss: 0.5445452928543091
epoch:  27, 

In [10]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -10739.6416015625
Test metrics:  R2 = -10640.724609375


In [11]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonRaphson(
    model=model,
    lr=1,
    c1=1e-4,
    c2=0.9,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="goldstein"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5523902177810669
epoch:  1, loss: 0.5523902177810669
epoch:  2, loss: 0.5523902177810669
epoch:  3, loss: 0.5523902177810669
epoch:  4, loss: 0.5523902177810669
epoch:  5, loss: 0.5523902177810669
epoch:  6, loss: 0.5523902177810669
epoch:  7, loss: 0.5523902177810669
epoch:  8, loss: 0.5523902177810669
epoch:  9, loss: 0.5523902177810669
epoch:  10, loss: 0.5523902177810669
epoch:  11, loss: 0.5523902177810669
epoch:  12, loss: 0.5523902177810669
epoch:  13, loss: 0.5523902177810669
epoch:  14, loss: 0.5523902177810669
epoch:  15, loss: 0.5523902177810669
epoch:  16, loss: 0.5523902177810669
epoch:  17, loss: 0.5523902177810669
epoch:  18, loss: 0.5523902177810669
epoch:  19, loss: 0.5523902177810669
epoch:  20, loss: 0.5523902177810669
epoch:  21, loss: 0.5523902177810669
epoch:  22, loss: 0.5523902177810669
epoch:  23, loss: 0.5523902177810669
epoch:  24, loss: 0.5523902177810669
epoch:  25, loss: 0.5523902177810669
epoch:  26, loss: 0.5523902177810669
epoch:  27,

In [12]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -400038.65625
Test metrics:  R2 = -406375.71875
